# Validation and Evaluation Plots for HAM10000 Baseline

This notebook loads the baseline artifacts from `reports/model_v1` and plots the main validation and evaluation visuals for the model.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Image

plt.style.use('seaborn-v0_8-whitegrid')
base_dir = Path('.')
metrics_path = base_dir / 'metrics_v1.json'
report_path = base_dir / 'classification_report_v1.csv'
cm_path = base_dir / 'confusion_matrix_v1.csv'
history_image_path = base_dir / 'training_history_v1.png'
compact_history_image_path = base_dir / 'training_history_compact_v1.png'

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
report_df = pd.read_csv(report_path, index_col=0)
cm_df = pd.read_csv(cm_path, index_col=0)

metrics

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Compact summary plot: key evaluation metrics
summary_items = [
    ('Recall melanoma', metrics.get('recall_melanoma', 0.0)),
    ('Recall macro', metrics.get('recall_macro', 0.0)),
    ('Precision macro', metrics.get('precision_macro', 0.0)),
    ('F1 macro', metrics.get('f1_macro', 0.0)),
    ('ROC-AUC OVR macro', metrics.get('roc_auc_ovr_macro', 0.0)),
]

summary_df = pd.DataFrame(summary_items, columns=['Metric', 'Value'])

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(summary_df['Metric'], summary_df['Value'], color='#4C78A8', edgecolor='black', linewidth=0.6)
ax.set_ylim(0, 1)
ax.set_title('Baseline Evaluation Summary', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.grid(True, axis='y', alpha=0.3)
ax.set_axisbelow(True)
plt.xticks(rotation=20, ha='right')

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.02,
        f'{height:.3f}',
        ha='center',
        va='bottom',
        fontsize=10,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(cm_df.values, cmap='Blues')
fig.colorbar(image, ax=ax)
ax.set_title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_xticks(np.arange(len(cm_df.columns)))
ax.set_yticks(np.arange(len(cm_df.index)))
ax.set_xticklabels(cm_df.columns, rotation=45, ha='right')
ax.set_yticklabels(cm_df.index)

for i in range(cm_df.shape[0]):
    for j in range(cm_df.shape[1]):
        ax.text(j, i, int(cm_df.iat[i, j]), ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class performance from the classification report
class_rows = [idx for idx in report_df.index if idx not in ['accuracy', 'macro avg', 'weighted avg']]
class_report = report_df.loc[class_rows, ['precision', 'recall', 'f1-score']]

ax = class_report.plot(kind='bar', figsize=(12, 5), width=0.8, color=['#4C78A8', '#F58518', '#54A24B'])
ax.set_title('Per-Class Metrics', fontsize=14, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.set_xticklabels(class_report.index, rotation=0)
ax.legend(loc='lower right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Display the saved training history figures if they exist
if history_image_path.exists():
    display(Image(filename=str(history_image_path)))

if compact_history_image_path.exists():
    display(Image(filename=str(compact_history_image_path)))